# 02 - Preprocessing & Feature Engineering
**Dataset:** NHANES 2017-2018 (post - EDA)

**Goal:** Handle missing values, encode categoricals, scale features, handle class imbalance, and save a model - ready dataset.

### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import pickle

print("Libraries Loaded")

Libraries Loaded


### Loading Data

In [2]:
df = pd.read_csv('../data/nhanes_after_eda.csv')
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Shape: (8187, 11)

Missing values:
WBC             833
RBC             833
Hemoglobin      833
Hematocrit      833
MCV             833
Platelets       833
Age               0
Gender            0
BMI             359
Diabetes          0
Gender_label      0
dtype: int64


### Dropping values where CBC values are Missing

In [3]:
cbc_cols = ['WBC', 'RBC', 'Hemoglobin', 'Hematocrit', 'MCV', 'Platelets']

before = len(df)
df = df.dropna(subset=cbc_cols)
after = len(df)

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Rows dropped: {before - after}")
print()
print("Reasoning: All CBC columns missing together = Participants")
print("Did not complete Blood draw. Imputing 6 values simultaneously")
print("Would be Clinically meaningless.")

Rows before: 8187
Rows after: 7354
Rows dropped: 833

Reasoning: All CBC columns missing together = Participants
Did not complete Blood draw. Imputing 6 values simultaneously
Would be Clinically meaningless.


### Imputing BMI with Median

In [4]:
bmi_median = df['BMI'].median()
df['BMI'] = df['BMI'].fillna(bmi_median)

print(f"BMI Missing values remaining: {df['BMI'].isnull().sum()}")
print(f"BMI imputed with median: {bmi_median:.2f}")

BMI Missing values remaining: 0
BMI imputed with median: 26.10


### Verifying No Missing values remains

In [5]:
print(f"Missing values after cleaning:")
print(df.isnull().sum())
print(f"\nFinal Shape: {df.shape}")

Missing values after cleaning:
WBC             0
RBC             0
Hemoglobin      0
Hematocrit      0
MCV             0
Platelets       0
Age             0
Gender          0
BMI             0
Diabetes        0
Gender_label    0
dtype: int64

Final Shape: (7354, 11)


### Encoding Gender

In [6]:
# Gender: 1 = Male, 2 = Female -> recode to 1 = Male, 0 = Female

df['Gender'] = df['Gender'].map({1.0: 1, 2.0: 0, 'Male': 1, 'Female': 0})
# Drop Gender_label if it exists (string column from EDA)
if 'Gender_label' in df.columns:
    df = df.drop(columns=['Gender_label'])
    print("Gender_label dropped")

print(df.dtypes)
print('Gender Encoded: 1 = Male, 0 = Female')
print(df['Gender'].value_counts())

Gender_label dropped
WBC           float64
RBC           float64
Hemoglobin    float64
Hematocrit    float64
MCV           float64
Platelets     float64
Age           float64
Gender          int64
BMI           float64
Diabetes        int64
dtype: object
Gender Encoded: 1 = Male, 0 = Female
Gender
0    3765
1    3589
Name: count, dtype: int64


### Checking Class Imbalance

In [7]:
diabetes_counts = df['Diabetes'].value_counts()
print("Diabetes distribution:")
print(diabetes_counts)
print(f"\nImbalance ratio: {round(diabetes_counts[0]/diabetes_counts[1], 2)}:1")
print(f"Diabetes rate: {round(df['Diabetes'].mean()*100,2)}%")

Diabetes distribution:
Diabetes
0    6536
1     818
Name: count, dtype: int64

Imbalance ratio: 7.99:1
Diabetes rate: 11.12%


### Feature Engineering

In [8]:
# Risk composite score - combines age, BMI and WBC into one signal
# Inspired by metabolic syndrome indicator used in clinical  settings
df['Age_BMI_interaction'] = df['Age'] * df['BMI']

# Anemia indicator — low hemoglobin is clinically associated
# diabetic complications (nephropathy affects erythropoiesis)
df['Anemia_flag'] = (df['Hemoglobin'] < 12.0).astype(int)

# High WBC flag — elevated WBC signals chronic inflammation,
# a known comorbidity of Type 2 diabetes
df['high_WBC_flag'] = (df['WBC'] > 11.0).astype(int)

print("Feature Engineering Completed")
print(f"New shape: {df.shape}")
print(f"\nAnemia cases flagged: {df['Anemia_flag'].sum()}")
print(f"\nHigh WBC cases flagged: {df['high_WBC_flag'].sum()}")

Feature Engineering Completed
New shape: (7354, 13)

Anemia cases flagged: 716

High WBC cases flagged: 432


### Seperate Features and Target

In [9]:
X = df.drop(columns=['Diabetes'])
y = df['Diabetes']

print(f"Feature Engineering: {X.shape}")
print(f"Target Shape: {y.shape}")
print(f"\nFeature Columns:\n{X.columns.tolist()}")

Feature Engineering: (7354, 12)
Target Shape: (7354,)

Feature Columns:
['WBC', 'RBC', 'Hemoglobin', 'Hematocrit', 'MCV', 'Platelets', 'Age', 'Gender', 'BMI', 'Age_BMI_interaction', 'Anemia_flag', 'high_WBC_flag']


### Saving Feature Columns

In [10]:
feature_columns = X.columns.tolist()

with open('../src/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)
    
print(f"Feature Columns Saved")
print(f"Total features: {len(feature_columns)}")
print(feature_columns)

Feature Columns Saved
Total features: 12
['WBC', 'RBC', 'Hemoglobin', 'Hematocrit', 'MCV', 'Platelets', 'Age', 'Gender', 'BMI', 'Age_BMI_interaction', 'Anemia_flag', 'high_WBC_flag']


### Scale Numerical Features

In [11]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Saving scaler for Streamlit App
with open('../src/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
print("Feature Scaled")
print("Scaler Saved")

Feature Scaled
Scaler Saved


### Saving Preprocessed Data

In [12]:
df_preprocessed = X_scaled.copy()
df_preprocessed['Diabetes'] = y.values

df_preprocessed.to_csv('../data/nhanes_preprocessed.csv', index=False)

print("Preprocessed Data Saved")
print(f"Final Shape: {df_preprocessed.shape}")

Preprocessed Data Saved
Final Shape: (7354, 13)


## Preprocessing Summary

**Missing Value Strategy:**
- CBC columns (10.17% missing): Dropped rows entirely
  → All 6 values missing together = participant skipped blood draw
  → Imputing 6 simultaneous blood values is clinically invalid
- BMI (4.39% missing): Imputed with median (single column, safe)

**Encoding:**
- Gender re-coded: 1=Male, 0=Female

**Class Imbalance:**
- Diabetes rate ~10% (7.99:1 ratio)
- Strategy: class_weight='balanced' in modeling (no SMOTE)

**Feature Engineering (domain-driven):**
- Age_BMI_interaction: metabolic syndrome composite signal
- Anemia_flag: Hemoglobin < 12 (diabetic nephropathy marker)
- High_WBC_flag: WBC > 11 (chronic inflammation marker)

**Output:** nhanes_preprocessed.csv ready for modeling